# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

##  Definição e Justificativa da Variável Alvo (Target)

Para treinar nosso modelo de classificação, precisamos estabelecer um critério claro do que define um "mau pagador" (*default*). Como nossa base de histórico financeiro (`credit_record.csv`) é longitudinal (mensal), construímos a variável alvo baseada nas seguintes premissas de negócio e gestão de risco:

* **O Limiar de Inadimplência:** A coluna `STATUS` classifica os pagamentos mensais. Optamos por definir como inadimplência grave (Target = 1) os status `2`, `3`, `4` e `5`, que representam faturas atrasadas há 60 dias ou mais. 
* **Justificativa de Negócio:** No setor bancário, atrasos curtos (menos de 60 dias) muitas vezes decorrem de esquecimentos ou desencontros de fluxo de caixa pontuais. Ultrapassar a barreira dos 60 dias indica uma degradação real na capacidade de pagamento, alinhando-se a métricas regulatórias de provisionamento de risco que exigem maior capitalização das instituições para absorver possíveis perdas.
* **A Lógica da Janela de Performance:** Ao agrupar o histórico por cliente (`ID`), aplicamos a função `.max()` na nossa variável temporária. Isso significa que adotamos uma postura conservadora: se o cliente atingiu o patamar de inadimplência grave em *qualquer momento* do seu histórico, ele é rotulado como `1` para o treinamento do modelo. Clientes que mantiveram status `0`, `1`, `C` (pago) ou `X` (sem empréstimo no mês) em todo o período recebem `0`.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Definir os caminhos e carregar os datasets brutos com pandas
path_app = Path("../data/raw/application_record.csv")
path_cred = Path("../data/raw/credit_record.csv")

df_app = pd.read_csv(path_app)
df_cred = pd.read_csv(path_cred)

print(f'Dimensão original do Cadastro (application_record.csv): {df_app.shape}')
print(f'Dimensão original do Histórico (credit_record.csv): {df_cred.shape}')

# 2. Construção da Variável Alvo (Target) baseada na regra de inadimplência (60+ dias de atraso)
maus_pagadores_status = ['2', '3', '4', '5']
df_cred['TARGET_TEMP'] = df_cred['STATUS'].astype(str).apply(
    lambda x: 1 if x in maus_pagadores_status else 0
)

# Agrupar por ID para capturar o pior status histórico do cliente (.max())
clientes_alvo = df_cred.groupby('ID')['TARGET_TEMP'].max().reset_index()
clientes_alvo.rename(columns={'TARGET_TEMP': 'TARGET'}, inplace=True)

# 3. Realizar o Inner Join para isolar a população de modelagem com histórico válido
df_modelagem = pd.merge(df_app, clientes_alvo, on='ID', how='inner')

print(f'\nDimensão da base unificada de modelagem (Inner Join): {df_modelagem.shape}')

# Cálculo e formatação da proporção da variável Target (2 casas decimais e símbolo %)
proporcoes = (df_modelagem['TARGET'].value_counts(normalize=True) * 100).round(2).astype(str) + '%'
print(f"Proporção da Target na população de modelagem:\n{proporcoes.to_string()}")

Dimensão original do Cadastro (application_record.csv): (438557, 18)
Dimensão original do Histórico (credit_record.csv): (1048575, 3)

Dimensão da base unificada de modelagem (Inner Join): (36457, 19)
Proporção da Target na população de modelagem:
TARGET
0    98.31%
1     1.69%


### Unificação das Bases (Inner Join) e Criação do Dataset de Modelagem

Para garantir a consistência dos dados, precisamos cruzar as informações demográficas do cadastro (`application_record.csv`) com a variável alvo financeira (`credit_record.csv`). 

A abordagem escolhida para essa consolidação foi o **Inner Join** (interseção) utilizando a coluna `ID` como chave de ligação. A justificativa para essa escolha é:

* **Validação Cruzada:** Ao utilizar o Inner Join, o código retém estritamente os clientes que aparecem **em ambas as tabelas simultaneamente**. 
* **Descarte de Registros Órfãos:** Clientes que possuem o histórico financeiro (Target), mas não o perfil cadastral (Features) — ou vice-versa — são sumariamente descartados. Para o treinamento de modelos de *Machine Learning*, registros sem o par completo de dados (perfil + alvo) não geram aprendizado e exigiriam imputações artificiais de dados nulos, comprometendo a precisão das regras capturadas.
* **Consolidação:** O resultado dessa interseção é armazenado no novo dataframe `df_modelagem`. Este *dataset* unificado representa a nossa população oficial e higienizada, sobre a qual todas as análises subsequentes, tratamentos de *outliers* e modelagens serão executadas.

A verificação final das proporções da variável alvo na base `df_modelagem` demonstra que, apesar do corte, a taxa original de representação da inadimplência manteve-se estável, indicando que o descarte de registros não causou viés na amostragem.

Um ponto importante: No gráfico de Balanceamento das Classes, gerado no primeiro notebook `01_eda.ipynb`, a quantidade de bons e maus pagadores classificados usa como base todo o dataset `credit_record.csv`. Na análise executada acima, os valores percentuais de **98,31%** (Target 0) e **1,68%** (Target 1) refletem a proporção estrita dentro do novo dataset `df_modelagem`. Isso significa que, dos 36.457 clientes resultantes da interseção, a maioria pertence à classe positiva de bons pagadores, evidenciando o forte desbalanceamento natural da base que precisará ser tratado nas etapas de modelagem.

## 1. Dados faltantes

In [3]:
# 1. Identificação de Dados Faltantes no dataset unificado
nulos = df_modelagem.isnull().sum()
percentual = (df_modelagem.isnull().mean() * 100).round(2)

df_nulos = pd.DataFrame({
    'Quantidade_Nulos': nulos, 
    'Percentual (%)': percentual
})

# Filtrar apenas as colunas que possuem algum valor vazio
colunas_com_nulos = df_nulos[df_nulos['Quantidade_Nulos'] > 0].sort_values(by='Percentual (%)', ascending=False)
print("Resumo de Dados Faltantes:\n")
print(colunas_com_nulos)

Resumo de Dados Faltantes:

                 Quantidade_Nulos  Percentual (%)
OCCUPATION_TYPE             11323           31.06


### 1.1. Diagnóstico e Decisão de Tratamento

A execução do bloco de código revelou que a variável `OCCUPATION_TYPE` (tipo de profissão) é a única que apresenta dados faltantes no nosso dataset unificado. Em formulários de concessão de crédito, a omissão desse dado costuma ocorrer quando o cliente atua na informalidade, não encontra sua área de atuação na lista de opções do sistema ou simplesmente prefere não declarar.

Para não perdermos esses clientes valiosos — que já possuem histórico financeiro mapeado na nossa variável alvo —, a decisão adotada foi preencher essas lacunas com a categoria `"Outros"`. Essa estratégia de imputação converte a ausência de dado em uma informação categórica explícita, permitindo que o algoritmo de *Machine Learning* interprete esse padrão de "não preenchimento" durante o treinamento, sem distorcer o peso das demais profissões.

In [4]:
# Substituir os valores nulos (NaN) da coluna OCCUPATION_TYPE pela string 'Outros'
df_modelagem['OCCUPATION_TYPE'] = df_modelagem['OCCUPATION_TYPE'].fillna('Outros')

# Validação do tratamento: somar todos os nulos do dataframe
total_nulos = df_modelagem.isnull().sum().sum()

print(f"Total de valores nulos na base após o tratamento: {total_nulos}")

Total de valores nulos na base após o tratamento: 0


Com a validação retornando zero, confirmamos que a base `df_modelagem` está totalmente higienizada em relação a dados faltantes. 

A estrutura agora está íntegra e pronta para a próxima fase do pré-processamento: a identificação e a padronização de *outliers* (valores extremos e atípicos), com foco especial na anomalia já detectada na coluna de tempo de emprego (`DAYS_EMPLOYED`).

## 2. Tratamento de Outliers (`DAYS_EMPLOYED`)

Durante a Análise Exploratória (Notebook 01), identificamos um comportamento anômalo na coluna `DAYS_EMPLOYED` (Tempo de Emprego): a presença do valor exato `365243`. 

Neste *dataset*, as datas relativas são representadas por números inteiros negativos (contagem de dias no passado até o momento da solicitação). O valor `365243` (que equivale matematicamente a exatos 1.000 anos no futuro) é um clássico *placeholder* (marcador de sistema padrão) utilizado pela instituição financeira para registrar clientes que não possuem emprego formal ativo no momento da aplicação — grupo que engloba aposentados, estudantes e desempregados. 

Essa interpretação sobre o valor `365243` é amplamente documentada na literatura de competições e análises deste *dataset* (como em referências de EDA do Home Credit Default Risk no Kaggle), onde o código indica a situação de inatividade laboral formal.

Se mantivermos esse valor na base, ele agirá como um *outlier* extremo e destruirá a escala da variável, prejudicando o desempenho de algoritmos de *Machine Learning* baseados em distância ou gradiente. A solução analítica mais coerente é substituir `365243` por `0`, representando corretamente que o cliente acumula "zero dias" de emprego formal atual.

In [5]:
# Verificar a quantidade de registros com o outlier antes do tratamento
qtd_outliers = (df_modelagem['DAYS_EMPLOYED'] == 365243).sum()
print(f"Clientes identificados com o marcador de sistema (365243): {qtd_outliers}")

# Substituir o outlier por 0
df_modelagem['DAYS_EMPLOYED'] = df_modelagem['DAYS_EMPLOYED'].replace(365243, 0)

# Validar o tratamento verificando o novo valor máximo da coluna
novo_max = df_modelagem['DAYS_EMPLOYED'].max()
print(f"Valor máximo em DAYS_EMPLOYED após o tratamento: {novo_max}")

Clientes identificados com o marcador de sistema (365243): 6135
Valor máximo em DAYS_EMPLOYED após o tratamento: 0


### Análise do Tratamento de Outliers

A execução do código confirmou a presença de **6.135 clientes** (aproximadamente 17% da nossa base de modelagem) registrados com o marcador de sistema `365243`. 

O sucesso do tratamento é evidenciado pela validação final, que retornou o valor máximo da coluna `DAYS_EMPLOYED` como **0**. Analiticamente, esse é o comportamento correto e esperado pela seguinte lógica:

* **Escala Temporal Coerente:** Como o banco de dados registra o tempo em dias contados retroativamente a partir da data de solicitação do crédito (valores negativos), o teto lógico para essa variável é exatamente `0`. Ele passa a representar fielmente os indivíduos que não possuem dias acumulados de emprego formal (desempregados ou aposentados).
* **Preservação do Algoritmo:** Ao substituir o salto artificial de 1.000 anos positivos por 0, garantimos que os algoritmos de *Machine Learning* interpretem a variável de forma linear e proporcional, sem que os cálculos de distância ou gradiente sejam severamente distorcidos por um *placeholder* do sistema.

## 3. Feature Engineering (Engenharia de Variáveis)

A Engenharia de Variáveis (*Feature Engineering*) é o processo analítico de transformar dados brutos em novos atributos que representem melhor o problema de negócio. O objetivo é facilitar o aprendizado do algoritmo de *Machine Learning*, tornando os padrões matemáticos mais evidentes e fáceis de processar.

Para o nosso modelo de risco de crédito, aplicaremos as seguintes estratégias baseadas nas características da nossa base de dados:

* **Conversão de Escala Temporal (Matemática Direta):** O sistema original registra tempo em dias negativos retroativos à data de solicitação. Transformaremos colunas como `DAYS_BIRTH` (idade do cliente) e `DAYS_EMPLOYED` (tempo de emprego) de dias para **anos** (valores positivos absolutos), tornando a grandeza linear e intuitiva.
* **Discretização (Criação de Faixas):** Converteremos variáveis contínuas muito granulares em grupos de negócio. Por exemplo, podemos transformar a renda exata (`AMT_INCOME_TOTAL`) em categorias de "Faixa de Renda".
* **Codificação de Categóricas (Encoding):** Variáveis em texto, como grau de escolaridade (`NAME_EDUCATION_TYPE`) ou posse de bens (`FLAG_OWN_CAR`), serão convertidas em representações numéricas binárias (0 e 1), um pré-requisito fundamental para que as equações do algoritmo consigam processar esses dados.

In [12]:
# Verificar e transformar 'DAYS_BIRTH' em 'IDADE_ANOS' apenas se a coluna original ainda existir
if 'DAYS_BIRTH' in df_modelagem.columns:
    df_modelagem['IDADE_ANOS'] = (df_modelagem['DAYS_BIRTH'] / -365.25).astype(int)

# Verificar e transformar 'DAYS_EMPLOYED' em 'TEMPO_EMPREGO_ANOS' apenas se a coluna original ainda existir
if 'DAYS_EMPLOYED' in df_modelagem.columns:
    df_modelagem['TEMPO_EMPREGO_ANOS'] = (df_modelagem['DAYS_EMPLOYED'] / -365.25)
    # Correção de precisão: Ajustar valores resultantes em -0.0 para 0 absoluto
    df_modelagem['TEMPO_EMPREGO_ANOS'] = df_modelagem['TEMPO_EMPREGO_ANOS'].apply(lambda x: 0.0 if x == -0.0 or x == 0.0 else x)
    df_modelagem['TEMPO_EMPREGO_ANOS'] = df_modelagem['TEMPO_EMPREGO_ANOS'].round(2)

# Excluir de forma segura as colunas originais em dias (caso ainda existam) para evitar redundância
colunas_dias_originais = ['DAYS_BIRTH', 'DAYS_EMPLOYED']
df_modelagem.drop(columns=[col for col in colunas_dias_originais if col in df_modelagem.columns], inplace=True)

# Visualizar as primeiras linhas para confirmar a transformação estrutural
print("Transformação temporal validada com sucesso.")
display(df_modelagem[['IDADE_ANOS', 'TEMPO_EMPREGO_ANOS']].head())

Transformação temporal validada com sucesso.


,IDADE_ANOS,TEMPO_EMPREGO_ANOS
0,32,12.435318
1,32,12.435318
2,58,3.104723
3,52,8.353183
4,52,8.353183


### 3.1 Execução das Transformações Temporais — Variáveis `DAYS_BIRTH` (Idade) e `DAYS_EMPLOYED` (Tempo de Emprego)

A primeira transformação aplicada foi a conversão das variáveis relativas `DAYS_BIRTH` (Idade) e `DAYS_EMPLOYED` (Tempo de Emprego). Ambas foram divididas por `-365.25` para transformá-las em anos positivos. A utilização do fator `365.25` é uma convenção analítica padrão para absorver a variação dos anos bissextos (calculada pela média de dias em um ciclo de 4 anos: $\frac{365 + 365 + 365 + 366}{4} = 365.25$), garantindo o cálculo exato e linear das idades.

Adicionalmente, após a criação das novas colunas `IDADE_ANOS` e `TEMPO_EMPREGO_ANOS`, as colunas originais formatadas em dias foram removidas do *dataset*. Essa exclusão obedece a uma regra técnica fundamental de engenharia de dados: evitar a **multicolinearidade**. Caso mantivéssemos a idade e o tempo de trabalho representados simultaneamente em dias e em anos na mesma base, duplicaríamos a mesma informação para os algoritmos de *Machine Learning*, o que distorceria o peso dessas características na avaliação preditiva do risco de crédito.

In [7]:
# 1. Traduzir as colunas originais para o português
df_modelagem.rename(columns={
    'AMT_INCOME_TOTAL': 'RENDA_TOTAL',
    'CNT_FAM_MEMBERS': 'QTD_MEMBROS_FAMILIA'
}, inplace=True)

# 2. Criação da variável de Renda Per Capita com as colunas traduzidas
df_modelagem['RENDA_PER_CAPITA'] = df_modelagem['RENDA_TOTAL'] / df_modelagem['QTD_MEMBROS_FAMILIA']

# 3. Arredondar para duas casas decimais
df_modelagem['RENDA_PER_CAPITA'] = df_modelagem['RENDA_PER_CAPITA'].round(2)

# 4. Visualizar as 5 primeiras linhas para confirmar a transformação
df_modelagem[['RENDA_TOTAL', 'QTD_MEMBROS_FAMILIA', 'RENDA_PER_CAPITA']].head()

,RENDA_TOTAL,QTD_MEMBROS_FAMILIA,RENDA_PER_CAPITA
0,427500.0,2.0,213750.0
1,427500.0,2.0,213750.0
2,112500.0,2.0,56250.0
3,270000.0,1.0,270000.0
4,270000.0,1.0,270000.0


### 3.2 Tradução e Criação de Variáveis de Proporção (Renda Per Capita)

No contexto de avaliação de risco de crédito, a renda bruta total (`RENDA_TOTAL`) analisada de forma isolada pode enviesar o modelo. Um solicitante com renda de R$ 10.000 que mora sozinho possui uma capacidade de pagamento e absorção de choques financeiros substancialmente diferente de um solicitante com a mesma renda que precisa sustentar uma família de cinco pessoas. 

Para capturar essa nuance econômica e entregar um padrão mais refinado ao algoritmo, primeiro traduzimos as variáveis originais para o português, visando facilitar a legibilidade do projeto. Em seguida, criamos a variável derivada `RENDA_PER_CAPITA`. Ao dividir a renda total pelo número de membros da família (`QTD_MEMBROS_FAMILIA`), transformamos um dado absoluto em um indicador de proporção que reflete com muito mais precisão o verdadeiro poder de compra e a margem de segurança financeira do cliente.

### 3.3 Codificação de Variáveis Categóricas (One-Hot Encoding)

Os algoritmos de *Machine Learning* realizam cálculos algébricos e, portanto, exigem que todas as entradas sejam estritamente numéricas. O nosso *dataset* ainda possui diversas colunas categóricas em formato de texto (como gênero, escolaridade, estado civil e posse de bens).

Para viabilizar a modelagem preditiva, aplicamos a técnica de *One-Hot Encoding* utilizando a função `pd.get_dummies()`. Esta abordagem transforma cada categoria de texto em uma nova coluna binária (0 ou 1). 

**Decisão Analítica Crítica:** Utilizamos o parâmetro `drop_first=True`. Isso remove a primeira categoria de cada variável original para evitar a **Armadilha da Variável Dummy** (*Dummy Variable Trap*). A exclusão previne a multicolinearidade perfeita, garantindo que o algoritmo não receba informações redundantes que poderiam inflar artificialmente os pesos do modelo durante o treinamento.

In [8]:
# 1. Converter flags binárias de texto ('Y'/'N') para numéricas (1/0)
binarias_yn = ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY']
for col in binarias_yn:
    if col in df_modelagem.columns:
        df_modelagem[col] = df_modelagem[col].apply(lambda x: 1 if str(x).strip().upper() == 'Y' else 0)

# 2. Converter a coluna de Gênero ('M'/'F') para binária (ex: GENERO_MASCULINO: 1 para M, 0 para F)
if 'CODE_GENDER' in df_modelagem.columns:
    df_modelagem['CODE_GENDER_M'] = df_modelagem['CODE_GENDER'].apply(lambda x: 1 if str(x).strip().upper() == 'M' else 0)
    df_modelagem.drop(columns=['CODE_GENDER'], inplace=True)

# 3. Identificar as demais colunas categóricas (tipo texto/'object') remanescentes
colunas_categoricas = df_modelagem.select_dtypes(include=['object']).columns
print(f"Colunas categóricas que serão codificadas:\n{list(colunas_categoricas)}\n")

# 4. Aplicar o One-Hot Encoding com drop_first=True para evitar multicolinearidade
df_modelagem = pd.get_dummies(df_modelagem, columns=colunas_categoricas, drop_first=True, dtype=int)

# 5. Tratamento de redundâncias (remover colunas originais em dias para evitar multicolinearidade com as versões em anos)
colunas_para_remover = ['DAYS_BIRTH', 'DAYS_EMPLOYED']
for col in colunas_para_remover:
    if col in df_modelagem.columns:
        df_modelagem.drop(columns=[col], inplace=True)

# Verificar a nova estrutura do dataset higienizado
print(f"Nova dimensão do dataset após o tratamento completo e Encoding: {df_modelagem.shape}")

# Visualizar as primeiras linhas com todas as colunas numéricas prontas
df_modelagem.head()

Colunas categóricas que serão codificadas:
['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']

Nova dimensão do dataset após o tratamento completo e Encoding: (36457, 50)


,ID,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,RENDA_TOTAL,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,QTD_MEMBROS_FAMILIA,...,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Outros,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Waiters/barmen staff
0,5008804,1,1,0,427500.0,1,1,0,0,2.0,...,0,0,0,1,0,0,0,0,0,0
1,5008805,1,1,0,427500.0,1,1,0,0,2.0,...,0,0,0,1,0,0,0,0,0,0
2,5008806,1,1,0,112500.0,1,0,0,0,2.0,...,0,0,0,0,0,0,0,0,1,0
3,5008808,0,1,0,270000.0,1,0,1,1,1.0,...,0,0,0,0,0,0,1,0,0,0
4,5008809,0,1,0,270000.0,1,0,1,1,1.0,...,0,0,0,0,0,0,1,0,0,0


### 3.3.3 Análise da Codificação de Variáveis

Nesta etapa, procedemos com a preparação integral do conjunto de dados para o ambiente de modelagem. Inicialmente, as flags binárias de texto (como a posse de carro e de imóvel) e a variável de gênero foram convertidas diretamente para o formato numérico binário (`0` e `1`). Em seguida, aplicamos o *One-Hot Encoding* nas demais variáveis categóricas plurais utilizando a função `pd.get_dummies()` com o parâmetro `drop_first=True`, o que evita a armadilha de variáveis *dummy* e previne a multicolinearidade.

Com essas transformações, o *dataset* foi integralmente convertido para um formato estritamente numérico, satisfazendo o pré-requisito fundamental para o treinamento e a estabilidade dos algoritmos de *Machine Learning*.

In [9]:
# Dicionário abrangente para traduzir prefixos e categorias do dataset original
dicionario_traducao = {
    # Traduções de colunas originais e prefixos
    'FLAG_MOBIL': 'TEM_TELEMÓVEL',
    'FLAG_WORK_PHONE': 'TEM_TELEFONE_TRABALHO',
    'FLAG_PHONE': 'TEM_TELEFONE_FIXO',
    'FLAG_EMAIL': 'TEM_EMAIL',
    'CODE_GENDER_M': 'GENERO_MASCULINO',
    'FLAG_OWN_CAR_Y': 'POSSUI_CARRO_SIM',
    'FLAG_OWN_REALTY_Y': 'POSSUI_IMOVEL_SIM',
    'NAME_INCOME_TYPE_': 'TIPO_RENDA_',
    'NAME_EDUCATION_TYPE_': 'ESCOLARIDADE_',
    'NAME_FAMILY_STATUS_': 'ESTADO_CIVIL_',
    'NAME_HOUSING_TYPE_': 'TIPO_HABITACAO_',
    'OCCUPATION_TYPE_': 'PROFISSAO_',
    
    # Traduções dos valores categóricos que viraram sufixos
    'Commercial associate': 'Associado_Comercial',
    'Pensioner': 'Pensionista',
    'State servant': 'Servidor_Publico',
    'Student': 'Estudante',
    'Working': 'Trabalhador_Ativo',
    'Higher education': 'Ensino_Superior',
    'Incomplete higher': 'Superior_Incompleto',
    'Lower secondary': 'Ensino_Fundamental',
    'Secondary / secondary special': 'Ensino_Medio',
    'Civil marriage': 'Casamento_Civil',
    'Married': 'Casado',
    'Separated': 'Separado',
    'Single / not married': 'Solteiro',
    'Widow': 'Viuvo',
    'Co-op apartment': 'Apto_Cooperativa',
    'House / apartment': 'Casa_Apto',
    'Municipal apartment': 'Apto_Municipal',
    'Office apartment': 'Apto_Escritorio',
    'Rented apartment': 'Apto_Alugado',
    'With parents': 'Com_Os_Pais'
}

# Iterar sobre o dicionário e substituir os termos nos nomes das colunas
novas_colunas = df_modelagem.columns
for termo_ingles, termo_portugues in dicionario_traducao.items():
    novas_colunas = novas_colunas.str.replace(termo_ingles, termo_portugues, regex=False)

# Aplicar os novos nomes ao dataset e remover espaços em branco por *underscores*
df_modelagem.columns = novas_colunas.str.replace(' ', '_')

# Visualizar a lista atualizada de colunas em formato de lista Python
print("Colunas formatadas e traduzidas:")
lista_colunas = df_modelagem.columns.tolist()
print(lista_colunas)

Colunas formatadas e traduzidas:
['ID', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'RENDA_TOTAL', 'TEM_TELEMÓVEL', 'TEM_TELEFONE_TRABALHO', 'TEM_TELEFONE_FIXO', 'TEM_EMAIL', 'QTD_MEMBROS_FAMILIA', 'TARGET', 'IDADE_ANOS', 'TEMPO_EMPREGO_ANOS', 'RENDA_PER_CAPITA', 'GENERO_MASCULINO', 'TIPO_RENDA_Pensionista', 'TIPO_RENDA_Servidor_Publico', 'TIPO_RENDA_Estudante', 'TIPO_RENDA_Trabalhador_Ativo', 'ESCOLARIDADE_Ensino_Superior', 'ESCOLARIDADE_Superior_Incompleto', 'ESCOLARIDADE_Ensino_Fundamental', 'ESCOLARIDADE_Ensino_Medio', 'ESTADO_CIVIL_Casado', 'ESTADO_CIVIL_Separado', 'ESTADO_CIVIL_Solteiro', 'ESTADO_CIVIL_Viuvo', 'TIPO_HABITACAO_Casa_Apto', 'TIPO_HABITACAO_Apto_Municipal', 'TIPO_HABITACAO_Apto_Escritorio', 'TIPO_HABITACAO_Apto_Alugado', 'TIPO_HABITACAO_Com_Os_Pais', 'PROFISSAO_Cleaning_staff', 'PROFISSAO_Cooking_staff', 'PROFISSAO_Core_staff', 'PROFISSAO_Drivers', 'PROFISSAO_HR_staff', 'PROFISSAO_High_skill_tech_staff', 'PROFISSAO_IT_staff', 'PROFISSAO_Laborers', 'PROFISSAO_L

### 3.5 Padronização e Tradução da Nomenclatura de Variáveis

O processo de *One-Hot Encoding* gerou dezenas de novas colunas combinando os nomes originais das variáveis e os seus respetivos valores em inglês (ex: `NAME_EDUCATION_TYPE_Higher education`). Esta nomenclatura dificulta a leitura do código, a governança dos dados e a futura interpretação da importância das variáveis (*Feature Importance*) por parte das equipas de negócio.

Para garantir a qualidade e a transparência do modelo de risco de crédito, aplicámos um processo de padronização léxica. As colunas e os seus sufixos foram traduzidos para português e os espaços em branco foram substituídos por *underscores* (`_`). Este tratamento assegura que todas as variáveis sigam as melhores práticas de engenharia de software e análise de dados, resultando num *dataset* limpo e intuitivo para a próxima fase de escalonamento.

## 4. Normalização / Padronização

**Escolha e justificativa:** A decisão técnica para esta etapa é **adiar a normalização/padronização para o Notebook 03 (Modelagem)**.

Embora algoritmos preditivos baseados em cálculos de distância ou gradiente (como Regressão Logística ou KNN) exijam que todas as variáveis (*features*) estejam na mesma grandeza, aplicar transformadores matemáticos (como `StandardScaler` ou `MinMaxScaler`) neste momento, sobre todo o *dataset* unificado, causaria um erro metodológico conhecido como **vazamento de dados** (*data leakage*).

Se ajustarmos o escalonador agora, ele aprenderá a distribuição matemática (média, variância, mínimos e máximos) considerando também os registros que futuramente comporão a base de teste. Isso enviesaria a avaliação, inflando artificialmente as métricas de acerto do modelo. O escalonamento será implementado de forma rigorosa na próxima fase, aplicando-se o `.fit()` exclusivamente sobre os dados de treino, preferencialmente dentro de um objeto `Pipeline` da biblioteca Scikit-Learn.

Existem variáveis com grandezas completamente diferentes, como a idade, que pode variar de 20 até 80 anos, enquanto a variável de rendimento pode variar de 1.000 até 100.000. Se normalizássemos as variáveis nesta etapa sobre todo o conjunto, os algoritmos de *Machine Learning* poderiam gerar resultados distorcidos, conferindo peso indevido a variáveis de maior escala numérica.

## 5. Salvar dataset tratado

In [10]:
from pathlib import Path

# Definir o caminho para o diretório de dados processados
PROCESSED = Path("../data/processed")

# Criar a pasta caso ela ainda não exista na estrutura do projeto
PROCESSED.mkdir(parents=True, exist_ok=True)

# Exportar o dataframe final para CSV (sem a coluna de índice do Pandas)
df_modelagem.to_csv(PROCESSED / "dataset_tratado.csv", index=False)

print(f"Dataset de modelagem salvo com sucesso em: {PROCESSED / 'dataset_tratado.csv'}")

Dataset de modelagem salvo com sucesso em: ..\data\processed\dataset_tratado.csv


### 5.1 Salvar Dataset Tratado

Com as etapas de limpeza de nulos, tratamento de *outliers* e *Feature Engineering* concluídas, a base de dados encontra-se consolidada e em formato estritamente numérico. 

O ficheiro final é exportado no formato `.csv` para o diretório `../data/processed/`. Este isolamento garante a integridade dos dados brutos originais e fornece um ponto de partida limpo e padronizado para a próxima fase do projeto (Notebook 03), onde os algoritmos de *Machine Learning* serão treinados e avaliados.